# UrbanSound8K datasheet

This datasheet documents the UrbanSound8K dataset as used by this repository. Every claim names its source; where no source answers, the entry says unknown. Nothing on this page is inferred.

Sources used:

- Dataset page: https://urbansounddataset.weebly.com/urbansound8k.html (referred to below as "dataset page")
- Dataset paper: J. Salamon, C. Jacoby and J. P. Bello, "A Dataset and Taxonomy for Urban Sound Research", 22nd ACM International Conference on Multimedia, Orlando, USA, Nov. 2014 (referred to as "dataset paper")
- Publications page of the dataset site: https://urbansounddataset.weebly.com/publications.html (referred to as "publications page")
- P1 audit report: IDIG4120_P1_Kivle_Stubø.ipynb (referred to as "P1 report")
- QA report: notebooks/02_quality_assurance.ipynb in this repository
- p2 documentation: p2/README.md and the module docstrings in p2/src

Datasheet reviewed by: Katharina Kivle and Nikolai O. Stubø.

## 1. Identity and purpose

| Field | Value | Source |
|---|---|---|
| Dataset name | UrbanSound8K | dataset page |
| Version | unknown, no version number is stated on the dataset page | dataset page |
| Creators | J. Salamon, C. Jacoby and J. P. Bello, the dataset paper authors | dataset paper |
| Contents | 8,732 labeled sound excerpts (at most 4 s) of urban sounds from 10 classes, in WAV format, plus a metadata CSV | dataset page |
| Unit of observation | one sound excerpt (a slice); file names follow [fsID]-[classID]-[occurrenceID]-[sliceID].wav | dataset page |
| Intended use | research on urban sound classification, evaluated with 10-fold cross validation on the predefined folds | dataset page |
| Use in this product | input material for the APAC compression pipeline; no classification is performed | this repository |
| Out-of-scope uses | safety-critical or production use; commercial reuse (see area 4); reshuffling the predefined folds when comparing with published classification results | our assessment; fold warning on the dataset page |

The ten classes: air_conditioner, car_horn, children_playing, dog_bark, drilling, engine_idling, gun_shot, jackhammer, siren, street_music (dataset page).

## 2. Collection and sampling

| Question | Answer | Source |
|---|---|---|
| Who or what is represented | urban sound events, recorded in the field and uploaded to freesound.org | dataset page |
| When | published with the dataset paper (ACM Multimedia 2014); the recording dates of the source material are not stated | dataset paper; unknown |
| How | excerpts were taken from field recordings; the metadata records each excerpt's start and end time inside its source recording | dataset page |
| Sampling frame | recordings available on freesound.org at collection time | dataset page |
| Selection and exclusion criteria | unknown; the dataset page refers to the paper for the compilation procedure and does not restate it | dataset page |
| Known gaps | geographic and temporal coverage of the recordings are not stated | unknown |
| Structure | excerpts pre-sorted into ten folds (fold1 to fold10) for comparable cross-validation results | dataset page |

## 3. Labels and transformations

- **Labels.** Every excerpt carries a class label in the metadata CSV, as classID 0-9 and a class name. The labels were produced by the dataset authors; the annotation procedure is described in the dataset paper and not restated on the dataset page, so its details are unknown here. Salience is a subjective rating: 1 = foreground, 2 = background (dataset page).
- **What this product changes on disk: nothing.** Files are used exactly as distributed; the manifest pins every file by SHA-256 digest.
- **What the loader changes in memory.** Every WAV subtype is converted to int16. Subtypes up to 16 bits are read directly; wider and float subtypes are read at native precision and quantized with round and clip, at most 1/2 LSB of error per sample (p2 loader documentation). No resampling: sample rate, bit depth and channel count stay as uploaded (dataset page).
- **Derived fields.** The pipeline computes compression metrics (sizes, ratio, RMSE, timing, memory) into output/results.json. No dataset fields are derived or stored.
- **The original data is kept: yes.** The pipeline writes only into the output directory.

## 4. Rights and privacy

| Question | Answer | Source |
|---|---|---|
| Licence | CC BY-NC 4.0 (non-commercial), as recorded from the Kaggle mirror during the P1 audit; the dataset site does not restate licence terms on its pages | P1 report; dataset page |
| Attribution | cite the dataset paper (see sources above); the dataset page asks that the dataset be acknowledged in academic research | publications page |
| Personal or sensitive data | unknown; the excerpts are field recordings of urban environments and the sources do not state whether identifiable persons are audible; the metadata CSV stores no personal fields | dataset page |
| Access | public download from the dataset site (web form or the soundata package) and from the Kaggle mirror | dataset page; P1 report |
| Retention and deletion | the working copy is kept for the duration of the course and deleted after grading | our policy |
| Retrieval of this copy | retrieved from the Kaggle mirror on 25 August 2026 | P1 report |

## 5. Quality and integrity

- **Units and valid ranges.** Excerpt duration at most 4 s; sample rate, bit depth and channels vary per file as uploaded (dataset page); salience 1 or 2; folds 1-10; classID 0-9 (dataset page). After loading: int16, shape (frames, channels), positive sample rate (see the schema section).
- **Measured values.** The QA report (notebook 02) records the counts from its run: verified digests, unreadable files, duplicate files, sample rates, channel counts, subtypes, durations, and the metadata CSV checks.
- **How missing values appear.** In the metadata CSV as empty cells; the QA check counts them per column when the CSV is present. In audio, an unreadable or invalid file surfaces as a load error.
- **Duplicates.** Byte-identical files are detected by comparing SHA-256 digests across all checked files; duplicate metadata rows by comparing full rows.
- **Manifest and hash checks.** Every file of the release has an entry with a relative path, byte size and SHA-256 digest in manifest/urbansound8k_manifest.json. `python run.py --verify-only` recomputes and compares them for every present file; `python run.py` verifies the input clip before any processing, and a digest mismatch aborts with exit code 3.

## 6. Use and maintenance

- **Versioning.** This repository is versioned with git. The manifest declares manifest_version 1.0. The dataset release itself carries no published version number (unknown).
- **Maintainers.** Katharina Kivle and Nikolai O. Stubø, for the duration of the IDIG4120 course.
- **Decision: use with controls.** Controls: digest verification before processing, int16 conversion at load, no resampling, the predefined folds kept intact. The dataset is not verified for label correctness, and the licence restricts commercial use.

In [1]:

import sys
from pathlib import Path

# find the repository root: the folder holding manifest/urbansound8k_manifest.json
root = Path.cwd()
while not (root / "manifest" / "urbansound8k_manifest.json").is_file():
    if root == root.parent:
        raise FileNotFoundError("run this notebook from inside the repository")
    root = root.parent
sys.path.insert(0, str(root / "src"))


from p3 import integrity

manifest = integrity.load_manifest(root / "manifest" / "urbansound8k_manifest.json")
files = manifest["files"]

per_top_level = {}
for entry in files:
    top = entry["path"].split("/")[0]
    per_top_level[top] = per_top_level.get(top, 0) + 1

print("dataset:", manifest["dataset"], "| manifest version:", manifest["manifest_version"])
print("total entries:", len(files))
for name in sorted(per_top_level):
    print(f"  {name}: {per_top_level[name]}")
print("total bytes covered by the manifest:", sum(f["size_bytes"] for f in files))


dataset: UrbanSound8K | manifest version: 1.0
total entries: 8733
  UrbanSound8K.csv: 1
  fold1: 873
  fold10: 837
  fold2: 888
  fold3: 925
  fold4: 990
  fold5: 936
  fold6: 823
  fold7: 838
  fold8: 806
  fold9: 816
total bytes covered by the manifest: 7089137662


The cell above is the datasheet's own evidence: the manifest covers the CSV plus all 8,732 excerpts, and the per-fold counts match the published fold sizes on the dataset page.

## Schema

Three layers have a contract: the metadata CSV, the audio after loading, and the APAC container the pipeline produces.

### Metadata CSV (UrbanSound8K.csv)

| Column | Type | Meaning | Valid range |
|---|---|---|---|
| slice_file_name | string | excerpt file name, format [fsID]-[classID]-[occurrenceID]-[sliceID].wav | matches one WAV file inside a fold |
| fsID | integer | Freesound ID of the source recording | unknown |
| start | number | excerpt start time in the source recording (s) | unknown |
| end | number | excerpt end time in the source recording (s) | end > start |
| salience | integer | subjective salience rating | 1 (foreground) or 2 (background) |
| fold | integer | predefined fold | 1-10 |
| classID | integer | sound class identifier | 0-9 |
| class | string | sound class name | one of the ten classes |

Source: dataset page.

### Audio contract after loading

| Property | Contract |
|---|---|
| dtype | int16 |
| shape | (frames, channels), 2D; at least one frame, at least one channel |
| sample rate | positive integer, taken from the file header |
| conversion | subtypes up to 16 bits read directly; wider and float subtypes read at native precision, then round and clip to int16, at most 1/2 LSB of error per sample |
| on disk | unchanged: no resampling, no rewriting |

Enforced by p3.pipeline.check_shape before any processing; a violation raises InvalidShape. Source: p2 loader and validation code.

### APAC container

33-byte big-endian header (magic "APAC", version 1, sample rate, frames, channels, predictor, block size, quantization step, DCT fields, entropy coder), then a data section of blocks, each starting with a control byte (raw, constant, or Rice-coded). The full specification is p2/src/audio_codec/codec/format.py. Source: p2 documentation.

## Known limitations of this product

- The working copy distributed with this repository carries fold1 (873 of 8,732 excerpts) inside p2/data/raw. The manifest covers the full release, so any complete copy can be verified and processed with the same commands.
- The metadata CSV is not part of this repository; the QA metadata checks run when it is present under the dataset dir and are reported as skipped otherwise.
- Timing and memory numbers come from one machine and one run; the method is the evidence, the recorded numbers are an example.
- No check verifies that the class labels describe the audio correctly.
- The licence (CC BY-NC 4.0, per the P1 audit) restricts commercial use of the dataset.

## Who did what

| Part | Who |
|---|---|
| Initial datasheet, QA report, manifest, first pipeline notebook | Katharina Kivle |
| Repository redo: pipeline code, entry point, integrity checks, tests, notebooks, README | Nikolai O. Stubø |
| Datasheet content, decisions, final review | both |

## Generative-AI declaration (P3)

Generative AI (Claude, through the Cline agent) was used while building this product: for the pipeline, integrity and test code, for the notebook structure and draft prose, and for refactoring. All numbers and outputs shown in the notebooks were produced by running the code in this repository on the dataset. The authors reviewed and edited the material before submission.